In [58]:
import os
os.makedirs("PDF/PDF_files",exist_ok=True)

In [59]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    PyMuPDFLoader
)

In [60]:
# Method 2: PyMuPDFLoader (Fast and accurate)
print("\n3️⃣ PyMuPDFLoader")
try:
    pymupdf_loader = PyMuPDFLoader("C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf")
    pymupdf_docs = pymupdf_loader.load()
    
    print(f"  Loaded {len(pymupdf_docs)} pages")
    print(f"  Includes detailed metadata")
    print(pymupdf_docs)
except Exception as e:
    print(f"  Error: {e}")


3️⃣ PyMuPDFLoader
  Loaded 764 pages
  Includes detailed metadata
[Document(metadata={'producer': 'PDFTron PDFNet, V7.10742', 'creator': '', 'creationdate': '2013-07-25T09:21:58+00:00', 'source': 'C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf', 'file_path': 'C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf', 'total_pages': 764, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2020-04-23T20:23:22+00:00', 'trapped': '', 'modDate': 'D:20200423202322Z', 'creationDate': 'D:20130725092158Z', 'page': 0}, page_content=''), Document(metadata={'producer': 'PDFTron PDFNet, V7.10742', 'creator': '', 'creationdate': '2013-07-25T09:21:58+00:00', 'source': 'C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf', 'file_path': 'C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf', 'total_pages': 764, 'format': 'PDF 1.4', 'title': '', 'author': '',

In [61]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from typing import List
import re
import unicodedata
import os

class SmartPDFProcessor:
    """Advanced PDF processing using PyMuPDFLoader with cleaning, chunking, and metadata."""

    def __init__(self, chunk_size=1000, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" ", "\n"]
        )

    def process_pdf(self, pdf_path: str) -> List[Document]:
        """Process a PDF into cleaned, chunked LangChain Documents."""
        try:
            loader = PyMuPDFLoader(pdf_path)
            pages = loader.load()
        except Exception as e:
            print(f"❌ Error loading {pdf_path}: {e}")
            return []

        processed_chunks = []

        for page_num, page in enumerate(pages):
            # Clean text
            cleaned_text = self._clean_text(page.page_content)

            # Skip nearly empty pages
            if len(cleaned_text.strip()) < 50:
                continue

            # Create chunks
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text),
                    "source_file": os.path.basename(pdf_path)
                }]
            )

            processed_chunks.extend(chunks)

        print(f"✅ Processed {len(processed_chunks)} chunks from '{os.path.basename(pdf_path)}'")
        return processed_chunks

    def process_directory(self, folder_path: str) -> List[Document]:
        """Process all PDFs in a directory and preview each one."""
        all_chunks = []
        pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]

        if not pdf_files:
            print("⚠️ No PDF files found in the directory.")
            return []

        print(f"\n📁 Found {len(pdf_files)} PDFs in '{folder_path}'\n")

        for pdf_file in pdf_files:
            pdf_path = os.path.join(folder_path, pdf_file)
            chunks = self.process_pdf(pdf_path)

            # Preview first 300 characters of first chunk
            if chunks:
                print(f"\n📘 Preview of '{pdf_file}':")
                print(chunks[0].page_content[:300], "...\n")

            all_chunks.extend(chunks)

        print(f"\n✅ Total Chunks Created from Directory: {len(all_chunks)}")
        return all_chunks

    def _clean_text(self, text: str) -> str:
        """Clean and normalize extracted text from PDFs."""
        # Normalize unicode
        text = unicodedata.normalize("NFKC", text)

        # Replace common ligatures/punctuation
        text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
        text = text.replace("’", "'").replace("‘", "'")
        text = text.replace("“", '"').replace("”", '"')
        text = text.replace("—", "-").replace("–", "-")

        # Remove non-printable/control characters
        text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)

        # Remove multiple spaces/tabs/newlines
        text = re.sub(r"\s+", " ", text).strip()

        return text

In [62]:
# Initialize processor
processor = SmartPDFProcessor(chunk_size=1000, chunk_overlap=100)

# Process a directory of PDFs and preview them
folder_path = "C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf"
all_chunks = processor.process_directory("C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf")

# Example: print metadata of first chunk
if all_chunks:
    print("\n🔍 First Chunk Metadata:")
    print(all_chunks[0].metadata)

NotADirectoryError: [WinError 267] The directory name is invalid: 'C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf'

In [ ]:
import os
import re
import unicodedata
from typing import List
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# -----------------------------
# SmartPDFProcessor Class
# -----------------------------
class SmartPDFProcessor:
    def __init__(self, chunk_size=1000, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=[" ", "\n"]
        )

    def process_pdf(self, pdf_path: str) -> List[Document]:
        loader = PyMuPDFLoader(pdf_path)
        pages = loader.load()
        processed_chunks = []

        for page_num, page in enumerate(pages):
            cleaned_text = self._clean_text(page.page_content)
            if len(cleaned_text.strip()) < 50:
                continue
            chunks = self.text_splitter.create_documents(
                texts=[cleaned_text],
                metadatas=[{
                    **page.metadata,
                    "page": page_num + 1,
                    "total_pages": len(pages),
                    "chunk_method": "smart_pdf_processor",
                    "char_count": len(cleaned_text),
                    "source_file": os.path.basename(pdf_path)
                }]
            )
            processed_chunks.extend(chunks)

        print(f"✅ Processed {len(processed_chunks)} chunks from '{os.path.basename(pdf_path)}'")
        return processed_chunks

    def process_directory(self, folder_path: str) -> List[Document]:
        all_chunks = []
        pdf_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".pdf")]
        if not pdf_files:
            print("⚠️ No PDF files found in the directory.")
            return []

        print(f"\n📁 Found {len(pdf_files)} PDFs in '{folder_path}'\n")
        for pdf_file in pdf_files:
            pdf_path = os.path.join(folder_path, pdf_file)
            chunks = self.process_pdf(pdf_path)
            if chunks:
                print(f"\n📘 Preview of '{pdf_file}':")
                print(chunks[0].page_content[:300], "...\n")
            all_chunks.extend(chunks)

        print(f"\n✅ Total Chunks Created from Directory: {len(all_chunks)}")
        return all_chunks

    def _clean_text(self, text: str) -> str:
        text = unicodedata.normalize("NFKC", text)
        text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
        text = text.replace("’", "'").replace("‘", "'")
        text = text.replace("“", '"').replace("”", '"')
        text = text.replace("—", "-").replace("–", "-")
        text = re.sub(r"[\x00-\x1f\x7f-\x9f]", " ", text)
        text = re.sub(r"\s+", " ", text).strip()
        return text

In [ ]:
folder_path = "C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf"

processor = SmartPDFProcessor(chunk_size=1000, chunk_overlap=100)
all_chunks = processor.process_directory("C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf")

# Preview first 2 chunks
for i, chunk in enumerate(all_chunks[:2]):
    print(f"\n📄 Chunk {i+1} (Page {chunk.metadata['page']} of {chunk.metadata['total_pages']}):")
    print(chunk.page_content[:300], "...")

NotADirectoryError: [WinError 267] The directory name is invalid: 'C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf'

In [57]:
folder_path = "C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf)"

processor = SmartPDFProcessor(chunk_size=1000, chunk_overlap=100)
all_chunks = processor.process_directory(folder_path)

# Preview first 2 chunks
for i, chunk in enumerate(all_chunks[:2]):
    print(f"\n📄 Chunk {i+1} (Page {chunk.metadata['page']} of {chunk.metadata['total_pages']}):")
    print(chunk.page_content[:300], "...")

FileNotFoundError: [WinError 3] The system cannot find the path specified: 'C:/Users/Himanshugarse/Rangkush/PDF/PDF_files/Basic Electrical Engineering.pdf)'

In [53]:
import chromadb
from langchain.vectorstores import Chroma

# Create a list of texts and corresponding metadatas
texts = [chunk.page_content for chunk in all_chunks]
metadatas = [chunk.metadata for chunk in all_chunks]

# Initialize Chroma DB (persistent directory)
persist_directory = "chroma_db"
vectorstore = Chroma.from_texts(
    texts=texts,
    embedding=embeddings,
    metadatas=metadatas,
    persist_directory=persist_directory
)

# Persist to disk
vectorstore.persist()
print(f"✅ Stored {len(texts)} chunks in Chroma DB at '{persist_directory}'")

NameError: name 'all_chunks' is not defined

In [56]:
texts = [chunk.page_content for chunk in all_chunks]

print("🔢 Generating embeddings for all chunks...")
chunk_embeddings = embeddings.embed_documents(texts)

print(f"✅ Generated embeddings for {len(chunk_embeddings)} chunks")
print(f"Each embedding vector has {len(chunk_embeddings[0])} dimensions")

# Preview first embedding vector (first 10 values)
print("\n🔹 Example embedding (first 10 values):")
print(chunk_embeddings[0][:10])

NameError: name 'all_chunks' is not defined